# Duka Electronics - H1 2026 Sales Exploration

*Half-year review (January-June 2026) prepared for the Head of Sales.*

## Executive Summary

- Duka Electronics earned **KES 30,755,900** from **1,000** orders in H1 2026 (average **KES 30,756** per order), and weekends contributed only **26.2%** of revenue.
- **Nairobi** is the top branch with **KES 9.80M** (31.9% of revenue), but the rankings differ: **Nakuru** takes **158** orders versus **139** for **Eldoret**, yet earns **KES 3.69M** against **KES 4.23M**.
- The top three products by revenue (Laptop 15in, Smartphone, Smart TV 55in) generate **68.4%** of revenue, while **Phone Charger** leads on volume with **707** units but only **KES 0.85M** in revenue.
- **2026-03** was the best month at **KES 6.41M** and **2026-02** the worst at **KES 4.65M**, a **37.6%** gap between them.
- Premium online orders from Nairobi and Mombasa total **33** orders worth **KES 2,586,000**, or **8.4%** of all revenue.

## 0. Setup

The data comes from `generate_data.py` (seed 2026), which writes `duka_sales.csv`. If the CSV is missing it is regenerated here.

In [1]:
import subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

if not Path("duka_sales.csv").exists():
    subprocess.run([sys.executable, "generate_data.py"], check=True)

pd.options.display.float_format = "{:,.2f}".format
KES = lambda v: f"KES {v:,.0f}"

## 1. Load the data and first-five-minutes inspection

In [2]:
# parse_dates converts order_date at load time, so it is a real datetime and not a string
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
display(df.head())
print("Shape:", df.shape)
print()
df.info()

,order_id,order_date,branch,product,channel,units,unit_price
0,DK0001,2026-01-01,Nakuru,Wireless Earbuds,In-store,1,6500
1,DK0002,2026-01-01,Nakuru,Bluetooth Speaker,Online,1,9500
2,DK0003,2026-01-01,Nairobi,Smartphone,In-store,2,22000
3,DK0004,2026-01-01,Nairobi,Tablet,In-store,3,18000
4,DK0005,2026-01-02,Kisumu,Microwave,Online,1,15000


Shape: (1000, 7)

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   str           
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   channel     1000 non-null   str           
 5   units       1000 non-null   int64         
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(2), str(4)
memory usage: 54.8 KB


In [3]:
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate order_id values:", df["order_id"].duplicated().sum())
print("Date range:", df["order_date"].min().date(), "->", df["order_date"].max().date())
print("\nNumeric summary:")
display(df[["units", "unit_price"]].describe())
for col in ["branch", "channel", "product"]:
    print(f"\n{col}:")
    print(df[col].value_counts())

Missing values per column:
order_id      0
order_date    0
branch        0
product       0
channel       0
units         0
unit_price    0
dtype: int64

Duplicate order_id values: 0
Date range: 2026-01-01 -> 2026-06-30

Numeric summary:


,units,unit_price
count,"1,000.00","1,000.00"
mean,2.11,"19,491.70"
std,1.49,"18,490.17"
min,1.00,"1,200.00"
25%,1.00,"6,500.00"
50%,2.00,"15,000.00"
75%,3.00,"22,000.00"
max,10.00,"62,000.00"



branch:
branch
Nairobi    307
Mombasa    208
Kisumu     188
Nakuru     158
Eldoret    139
Name: count, dtype: int64

channel:
channel
In-store    650
Online      350
Name: count, dtype: int64

product:
product
Phone Charger        191
Smartphone           190
Wireless Earbuds     144
Bluetooth Speaker    119
Tablet               102
Smart TV 55in         98
Laptop 15in           88
Microwave             68
Name: count, dtype: int64


### Inspection findings

- **Shape:** 1,000 rows x 7 raw columns (12 after the derived columns are added).
- **Types:** `order_date` is parsed as `datetime64` (thanks to `parse_dates`); `branch`, `product`, `channel` and `order_id` are text; `units` and `unit_price` are integers.
- **Missing values:** none in any column, and `order_id` has no duplicates, so no cleaning was needed.
- **Date range:** 01 Jan 2026 to 30 Jun 2026, exactly the half-year under review.
- **Key categories:** 5 branches (Eldoret, Kisumu, Mombasa, Nairobi, Nakuru), 2 channels (In-store, Online) and 8 products.
- **Revenue is not in the raw data.** It has to be derived as `units x unit_price` (next section).

## 2. Derived columns

`revenue = units x unit_price`; `month` as `YYYY-MM` (sorts chronologically); `weekday` as the day name; `is_weekend` is True for Saturday and Sunday; `price_band` is **Premium** when `unit_price` >= KES 28,000 and **Value** otherwise.

*Assumption:* the Premium threshold is applied to the **unit price** of the item, not the order revenue (a customer buying five chargers is still buying a Value product).

In [4]:
df["revenue"] = df["units"] * df["unit_price"]
df["month"] = df["order_date"].dt.strftime("%Y-%m")
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

display(df.head())
print("Total revenue:", KES(df["revenue"].sum()), "| orders:", len(df))
print(df["price_band"].value_counts())

,order_id,order_date,branch,product,channel,units,unit_price,revenue,month,weekday,is_weekend,price_band
0,DK0001,2026-01-01,Nakuru,Wireless Earbuds,In-store,1,6500,6500,2026-01,Thursday,False,Value
1,DK0002,2026-01-01,Nakuru,Bluetooth Speaker,Online,1,9500,9500,2026-01,Thursday,False,Value
2,DK0003,2026-01-01,Nairobi,Smartphone,In-store,2,22000,44000,2026-01,Thursday,False,Value
3,DK0004,2026-01-01,Nairobi,Tablet,In-store,3,18000,54000,2026-01,Thursday,False,Value
4,DK0005,2026-01-02,Kisumu,Microwave,Online,1,15000,15000,2026-01,Friday,False,Value


Total revenue: KES 30,755,900 | orders: 1000
price_band
Value      814
Premium    186
Name: count, dtype: int64


## 3. Task 3 - Branch ranking by revenue and by orders

In [5]:
branch = df.groupby("branch").agg(total_revenue=("revenue", "sum"), orders=("order_id", "count"))
branch["revenue_rank"] = branch["total_revenue"].rank(ascending=False).astype(int)
branch["orders_rank"] = branch["orders"].rank(ascending=False).astype(int)
branch["avg_order_value"] = branch["total_revenue"] / branch["orders"]
branch = branch.sort_values("total_revenue", ascending=False)
display(branch)
print("Rankings identical?", (branch["revenue_rank"] == branch["orders_rank"]).all())

,total_revenue,orders,revenue_rank,orders_rank,avg_order_value
branch,,,,,
Nairobi,9797100,307,1,1,"31,912.38"
Mombasa,7068500,208,2,2,"33,983.17"
Kisumu,5965300,188,3,3,"31,730.32"
Eldoret,4231700,139,4,5,"30,443.88"
Nakuru,3693300,158,5,4,"23,375.32"


Rankings identical? False


**Answer:** the rankings are **not** the same. Nairobi is first on both lists, but **Nakuru** has more orders (158) than **Eldoret** (139) while earning less revenue (KES 3,693,300 vs KES 4,231,700). The reason is order value: Eldoret averages KES 30,444 per order against KES 23,375 for Nakuru, so its customers buy more expensive products (or more units). Revenue depends on *what* is sold, not just *how many* orders come in, which is why order counts alone cannot rank branches.

## 4. Task 4 - Top three products by revenue and by units

In [6]:
product = df.groupby("product").agg(total_revenue=("revenue", "sum"), units_sold=("units", "sum"))
top_rev = product.nlargest(3, "total_revenue")
top_units = product.nlargest(3, "units_sold")
print("Top 3 by revenue:"); display(top_rev)
print("Top 3 by units sold:"); display(top_units)
print("Same products?", set(top_rev.index) == set(top_units.index))

Top 3 by revenue:


,total_revenue,units_sold
product,,
Laptop 15in,7750000,125
Smartphone,6996000,318
Smart TV 55in,6288000,131


Top 3 by units sold:


,total_revenue,units_sold
product,,
Phone Charger,848400,707
Wireless Earbuds,2119000,326
Smartphone,6996000,318


Same products? False


**Answer:** by revenue the top three are **Laptop 15in**, **Smartphone** and **Smart TV 55in**; by units they are **Phone Charger** (707), **Wireless Earbuds** (326) and **Smartphone** (318).
The lists differ because price varies enormously. Phone Charger, Wireless Earbuds sell in high volume at low prices (a Phone Charger costs KES 1,200), so they dominate unit counts but add little revenue, while the Laptop 15in (KES 62,000) earns KES 7.75M from just 125 units. For stock planning, units matter; for profit and targets, revenue matters.

## 5. Task 5 - Weekend vs weekday

In [7]:
wk = (df.groupby("is_weekend")
        .agg(orders=("order_id", "count"), total_revenue=("revenue", "sum"), avg_order_revenue=("revenue", "mean"))
        .rename(index={False: "Weekday", True: "Weekend"}))
display(wk)
n_days = df.groupby("is_weekend")["order_date"].nunique().rename(index={False: "Weekday", True: "Weekend"})
print("Revenue per calendar day:")
print((wk["total_revenue"] / n_days).round(0))

,orders,total_revenue,avg_order_revenue
is_weekend,,,
Weekday,721,22691600,"31,472.40"
Weekend,279,8064300,"28,904.30"


Revenue per calendar day:
is_weekend
Weekday   177,278.00
Weekend   155,083.00
dtype: float64


**Answer:** weekdays account for **721** orders and **KES 22,691,600** (average **KES 31,472** per order); weekends account for **279** orders and **KES 8,064,300** (average **KES 28,904** per order).
Weekends are 26.2% of revenue, which is close to their share of the calendar (2 of 7 days = 28.6%), so weekend trading is not meaningfully stronger or weaker per day. The average order value is lower on weekends by 8.2%.

## 6. Task 6 - Best and worst month

In [8]:
monthly = df.groupby("month")["revenue"].sum()
display(monthly.to_frame("total_revenue"))
best, worst = monthly.idxmax(), monthly.idxmin()
pct = (monthly.max() - monthly.min()) / monthly.min() * 100
print(f"Best month : {best}  {KES(monthly.max())}")
print(f"Worst month: {worst}  {KES(monthly.min())}")
print(f"Best is {pct:.1f}% higher than worst")

,total_revenue
month,
2026-01,4883200
2026-02,4654400
2026-03,6405800
2026-04,5135000
2026-05,4798400
2026-06,4879100


Best month : 2026-03  KES 6,405,800
Worst month: 2026-02  KES 4,654,400
Best is 37.6% higher than worst


**Answer:** **2026-03** is the best month with **KES 6,405,800** and **2026-02** the worst with **KES 4,654,400**. The best month is **37.6% higher** than the worst, calculated as (best - worst) / worst. (Measured against the best month instead, the worst is 27.3% lower.)

## 7. Task 7 - Premium online orders from Nairobi and Mombasa

In [9]:
mask = (
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
)
premium_online = df[mask]
print("Orders:", len(premium_online))
print("Total revenue:", KES(premium_online["revenue"].sum()))
display(premium_online.groupby("branch")["revenue"].agg(["count", "sum"]))

Path("outputs").mkdir(exist_ok=True)
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)
print("Saved outputs/premium_online_coast_capital.csv")

Orders: 33
Total revenue: KES 2,586,000


,count,sum
branch,,
Mombasa,17,1218000
Nairobi,16,1368000


Saved outputs/premium_online_coast_capital.csv


**Answer:** there are **33** Premium online orders from Nairobi and Mombasa, worth **KES 2,586,000** in total (8.4% of all revenue). They have been exported to `outputs/premium_online_coast_capital.csv` with `index=False`.

## 8. Task 8 - Spreadsheet check (Excel vs pandas)

`excel/duka_branch_summary.xlsx` holds the SUMIFS branch table (sheet `BranchSummary`), the XLOOKUP region column, and a PivotTable of revenue by branch and channel (sheet `PivotTable`). The cell below reads the workbook and compares its totals with pandas.

In [10]:
import openpyxl

wb = openpyxl.load_workbook("excel/duka_branch_summary.xlsx", data_only=True)
bs = wb["BranchSummary"]
xl = pd.DataFrame(
    [[r[0], r[1], r[2], r[3]] for r in bs.iter_rows(min_row=2, max_row=1 + df["branch"].nunique(), values_only=True)],
    columns=["branch", "region", "xl_orders", "xl_revenue"],
).set_index("branch")
check = xl.join(branch[["orders", "total_revenue"]])
check["revenue_match"] = check["xl_revenue"] == check["total_revenue"]
check["orders_match"] = check["xl_orders"] == check["orders"]
display(check)

# PivotTable: read the branch x channel grid and its grand total
pv = wb["PivotTable"]
rows = [[c for c in r] for r in pv.iter_rows(min_row=3, values_only=True)]
pivot = pd.DataFrame([r for r in rows if r[0] is not None])
display(pivot)

pandas_grid = df.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum")
grand = pandas_grid.to_numpy().sum()
pivot_numbers = [v for r in rows for v in r[1:] if isinstance(v, (int, float))]
print("pandas grand total :", KES(grand))
print("Excel SUMIFS total :", KES(xl["xl_revenue"].sum()))
print("PivotTable total   :", KES(max(pivot_numbers)))
assert check["revenue_match"].all() and check["orders_match"].all()
assert xl["xl_revenue"].sum() == grand == max(pivot_numbers)
print("All Excel totals match pandas.")
display(pandas_grid)

,region,xl_orders,xl_revenue,orders,total_revenue,revenue_match,orders_match
branch,,,,,,,
Nairobi,Central,307,9797100,307,9797100,True,True
Mombasa,Coast,208,7068500,208,7068500,True,True
Kisumu,Western,188,5965300,188,5965300,True,True
Eldoret,Rift Valley,139,4231700,139,4231700,True,True
Nakuru,Rift Valley,158,3693300,158,3693300,True,True


,0,1,2,3
0,Filter,None,None,None
1,Sum - revenue,channel,None,None
2,branch,In-store,Online,Total Result
3,Eldoret,2789700,1442000,4231700
4,Kisumu,3367900,2597400,5965300
5,Mombasa,4786500,2282000,7068500
6,Nairobi,6716500,3080600,9797100
7,Nakuru,2241200,1452100,3693300
8,Total Result,19901800,10854100,30755900


pandas grand total : KES 30,755,900
Excel SUMIFS total : KES 30,755,900
PivotTable total   : KES 30,755,900
All Excel totals match pandas.


channel,In-store,Online
branch,,
Eldoret,2789700,1442000
Kisumu,3367900,2597400
Mombasa,4786500,2282000
Nairobi,6716500,3080600
Nakuru,2241200,1452100


**Answer:** every branch's SUMIFS revenue and COUNTIFS order count equals the pandas figure, and the PivotTable grand total equals both, so the spreadsheet and the notebook agree to the shilling.

## 9. Conclusion

The five-sentence **Executive Summary** at the top of this notebook is the answer for leadership; sections 3-8 hold the supporting code and working.